# 02 — Fragmentomic features

What each feature block actually looks like, and whether the differences
between groups are the ones the literature predicts.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../src'))
import numpy as np, pandas as pd
from cfdna_insight.features import load_features

FEATURES = '../results/features'   # or '../results/smoke/features'
blocks = load_features(FEATURES)
samples = pd.read_csv(os.path.join(FEATURES, 'samples_used.tsv'), sep='\t')
order = [s for s in samples['sample_id'] if s in blocks['size_summary'].index]
samples = samples.set_index('sample_id').loc[order].reset_index()
y = (samples['label'] == 'cancer').astype(int).to_numpy()
{k: v.shape for k, v in blocks.items()}

## Size summaries

Expected direction: cancer shifts shorter. If the effect runs the other
way, suspect pre-analytical variation — white-cell lysis in the tube
releases longer, less protected DNA.

In [ ]:
summ = blocks['size_summary'].reindex(order)
from scipy.stats import mannwhitneyu
rows = []
for col in summ.columns:
    v = summ[col].to_numpy(dtype=float); ok = np.isfinite(v)
    if ok.sum() < len(v) * 0.8: continue
    a, b = v[ok & (y == 1)], v[ok & (y == 0)]
    u, p = mannwhitneyu(a, b)
    rows.append({'feature': col, 'cancer_median': np.median(a),
                 'healthy_median': np.median(b), 'p': p})
res = pd.DataFrame(rows).sort_values('p')
res['p_bonferroni'] = (res['p'] * len(res)).clip(upper=1)
res

## The DELFI profile

Rows are samples, columns are 5 Mb windows in genome order. What to look
for is not a uniform colour shift — the within-sample z-scoring removes
that by construction — but whether cancer rows are *more variable* along
the genome than healthy rows.

In [ ]:
from cfdna_insight.plots import plot_delfi_heatmap
plot_delfi_heatmap(blocks['delfi'].reindex(order), y, '/tmp/delfi.png')
from IPython.display import Image; Image('/tmp/delfi.png')

In [ ]:
delfi = blocks['delfi'].reindex(order)
profile_sd = delfi.std(axis=1)
u, p = mannwhitneyu(profile_sd[y == 1], profile_sd[y == 0])
print(f'profile variability, cancer {profile_sd[y==1].median():.3f} vs '
      f'healthy {profile_sd[y==0].median():.3f}, p = {p:.3g}')

## End motifs

Only present if a reference FASTA was supplied. Reduced motif diversity
in cancer is the expected direction, reflecting reduced DNASE1L3 activity.

In [ ]:
if 'motif' in blocks:
    mds = blocks['size_summary'].reindex(order)['mds']
    u, p = mannwhitneyu(mds[y == 1].dropna(), mds[y == 0].dropna())
    print(f'MDS cancer {mds[y==1].median():.4f} vs healthy '
          f'{mds[y==0].median():.4f}, p = {p:.3g}')
    motifs = blocks['motif'].reindex(order)
    diff = (motifs[y == 1].mean() - motifs[y == 0].mean()).sort_values()
    print('\nmost shifted 4-mers:'); print(pd.concat([diff.head(5), diff.tail(5)]))
else:
    print('no motif block: rerun feature extraction with --fasta')